In [ ]:
import json
import sys
from pathlib import Path

# Resolve project root and ensure local package imports work from notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / "nepse_analyst").exists():
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BENCHMARK_PATH = PROJECT_ROOT / "evaluation" / "benchmark_questions.json"
REPORTS_DIR = PROJECT_ROOT / "evaluation" / "results"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

from nepse_analyst.sql_generator import generate_and_execute

In [ ]:
with open(BENCHMARK_PATH) as f:
    benchmark = json.load(f)

questions = benchmark["sql_benchmark"]
print(f"Loaded {len(questions)} benchmark questions")

In [ ]:
results = []

for q in questions:
    print(f"\n{'='*60}")
    print(f"[{q['id']}] {q['question']}")
    
    result = generate_and_execute(q['question'])
    
    print(f"SQL generated (attempt {result['attempts']}):")
    print(result['sql'])
    print(f"Rows returned: {result['row_count']}")
    if result['rows']:
        print(f"First row: {result['rows'][0]}")
    if not result['success']:
        print(f"ERROR: {result['error']}")
    
    results.append({
        "id": q['id'],
        "question": q['question'],
        "difficulty": q['difficulty'],
        "ground_truth": q['ground_truth'],
        "generated_sql": result['sql'],
        "returned_rows": result['rows'][:3],  # save first 3 rows
        "row_count": result['row_count'],
        "attempts": result['attempts'],
        "success": result['success'],
        "error": result.get('error'),
        "score": None,  # to be filled manually below
        "notes": ""
    })

In [ ]:
# Fill in scores manually after reviewing each result above
scores = {
    "Q1":  1.0,   # correct — NABIL with correct EPS
    "Q2":  0.5,   # partial — right companies but missing one
    "Q3":  1.0,
    # ... fill in all 12
}

for r in results:
    r['score'] = scores.get(r['id'], 0.0)

total = sum(r['score'] for r in results)
print(f"\nTotal score: {total:.1f} / {len(results)}.0")
print(f"Accuracy: {total/len(results)*100:.1f}%")

# By difficulty
for diff in ['easy', 'medium', 'hard']:
    subset = [r for r in results if r['difficulty'] == diff]
    s = sum(r['score'] for r in subset)
    print(f"  {diff.capitalize()}: {s:.1f}/{len(subset)}.0")

In [ ]:
from datetime import datetime

report = {
    "run_date": datetime.now().isoformat(),
    "total_score": total,
    "max_score": len(results),
    "accuracy_pct": round(total / len(results) * 100, 1),
    "pass_threshold": 8.0,
    "passed": total >= 8.0,
    "results": results,
}

report_path = REPORTS_DIR / f"eval_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Report saved to {report_path}")

In [ ]:
from nepse_analyst.retriever import search
from nepse_analyst.prompts import build_rag_synthesis_prompt
from nepse_analyst import llm
from nepse_analyst.language_detector import detect_language

rag_questions = [
    {"id": "R1", "question": "What recent news is there about Nabil Bank?"},
    {"id": "R2", "question": "Are there any upcoming AGMs announced in the banking sector?"},
    {"id": "R3", "question": "What is the regulatory environment for hydropower IPOs in Nepal currently?"},
    {"id": "R4", "question": "Summarise the latest quarterly earnings news for major commercial banks."},
    {"id": "R5", "question": "कुन कम्पनीले हालसालै बोनस सेयर घोषणा गर्यो?"},
]

rag_results = []

for q in rag_questions:
    print(f"\n{'='*60}\n[{q['id']}] {q['question']}")
    
    passages = search(q['question'], top_k=5)
    query_lang = detect_language(q['question'])
    
    print(f"Top {len(passages)} passages retrieved:")
    for p in passages:
        print(f"  [{p['relevance_score']:.3f}] {p['published_at']} | {p['title'][:70]}")
    
    if passages:
        prompt = build_rag_synthesis_prompt(q['question'], passages, query_lang)
        answer = llm.call(prompt, temperature=0.1)
        print(f"\nSynthesised answer:\n{answer[:400]}")
    else:
        answer = "No relevant passages found in the corpus."
        print("No passages retrieved.")
    
    rag_results.append({
        "id": q['id'],
        "question": q['question'],
        "query_language": query_lang,
        "passages_retrieved": len(passages),
        "top_passage_title": passages[0]['title'] if passages else "",
        "top_relevance_score": passages[0]['relevance_score'] if passages else 0.0,
        "synthesised_answer": answer,
        "top_passage_relevant": None,   # fill in manually: True/False
        "notes": ""
    })